# **Fantasy Premier League Data Loader**
This notebook connects to the official FPL API, loads the static core data into pandas DataFrames, and includes helpers for player history/fixture analysis.

### **1. Data Loading**

In [2]:
import time
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
import requests

BASE_URL = "https://fantasy.premierleague.com/api"


def fetch_json(url: str, timeout: int = 20) -> Optional[Dict[str, Any]]:
    """Fetch JSON from a URL with basic error handling."""
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        print(f"Request timed out for: {url}")
    except requests.exceptions.HTTPError as exc:
        print(f"HTTP error for {url}: {exc}")
    except requests.exceptions.RequestException as exc:
        print(f"Request failed for {url}: {exc}")
    except ValueError as exc:
        print(f"Invalid JSON returned by {url}: {exc}")
    return None


def load_bootstrap_data() -> Dict[str, Any]:
    """Fetch and return the static bootstrap data from the official FPL API."""
    bootstrap = fetch_json(f"{BASE_URL}/bootstrap-static/")
    if bootstrap is None:
        raise RuntimeError("Unable to load FPL bootstrap-static data.")
    return bootstrap


def build_core_dataframes() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build players_df, teams_df and positions_df from the bootstrap data."""
    bootstrap = load_bootstrap_data()
    elements = bootstrap.get("elements", [])
    teams = bootstrap.get("teams", [])
    positions = bootstrap.get("element_types", [])

    team_map = {team.get("id"): team.get("name") for team in teams if team.get("id") is not None}
    position_map = {
        pos.get("id"): pos.get("singular_name_short") or pos.get("singular_name") or pos.get("name")
        for pos in positions
        if pos.get("id") is not None
    }

    # Players table: use readable team and position labels while keeping raw IDs for joins
    players_df = pd.DataFrame(elements)
    if not players_df.empty:
        players_df["team_id"] = players_df.get("team", pd.Series(dtype="float64"))
        players_df["team"] = players_df["team_id"].map(team_map)

        players_df["position_id"] = players_df.get("element_type", pd.Series(dtype="float64"))
        players_df["position"] = players_df["position_id"].map(position_map)

        players_df["price"] = pd.to_numeric(players_df.get("now_cost", 0), errors="coerce") / 10.0
        players_df["xg"] = pd.to_numeric(players_df.get("expected_goals", 0), errors="coerce")
        players_df["xa"] = pd.to_numeric(players_df.get("expected_assists", 0), errors="coerce")
        players_df["ict_index"] = pd.to_numeric(players_df.get("ict_index", 0), errors="coerce")
        players_df["minutes"] = pd.to_numeric(players_df.get("minutes", 0), errors="coerce")
        players_df["selected_by_percent"] = pd.to_numeric(players_df.get("selected_by_percent", 0), errors="coerce")
        players_df["status"] = players_df.get("status", "").fillna("unknown")
        players_df["availability"] = players_df["status"]

    players_df = players_df.rename(columns={
        "id": "player_id",
        "web_name": "name",
        "total_points": "total_points",
        "form": "form",
        "minutes": "minutes",
        "selected_by_percent": "ownership_pct",
        "status": "status",
    })

    players_df = players_df[
        [
            "player_id",
            "name",
            "team_id",
            "team",
            "position_id",
            "position",
            "price",
            "total_points",
            "form",
            "xg",
            "xa",
            "ict_index",
            "minutes",
            "ownership_pct",
            "status",
            "availability",
        ]
    ]

    teams_df = pd.DataFrame(teams)
    if not teams_df.empty:
        teams_df = teams_df.rename(columns={
            "id": "team_id",
            "name": "team_name",
            "short_name": "short_name",
            "strength_overall_home": "strength_home",
            "strength_overall_away": "strength_away",
            "strength_attack_home": "attack_strength_home",
            "strength_attack_away": "attack_strength_away",
            "strength_defence_home": "defence_strength_home",
            "strength_defence_away": "defence_strength_away",
        })
        teams_df = teams_df[[
            "team_id",
            "team_name",
            "short_name",
            "strength_home",
            "strength_away",
            "attack_strength_home",
            "attack_strength_away",
            "defence_strength_home",
            "defence_strength_away",
        ]]

    positions_df = pd.DataFrame(positions)
    if not positions_df.empty:
        positions_df = positions_df.rename(columns={
            "id": "position_id",
            "singular_name": "position_name",
            "singular_name_short": "position_short",
            "name": "position_name",
        })
        positions_df = positions_df[["position_id", "position_name", "position_short"]]

    return players_df, teams_df, positions_df


def load_fixtures(next_gameweeks: int = 5) -> pd.DataFrame:
    """Fetch upcoming fixtures and return a team-by-team FDR table for the next few gameweeks."""
    bootstrap = load_bootstrap_data()
    team_map = {team.get("id"): team.get("name") for team in bootstrap.get("teams", []) if team.get("id") is not None}
    fixtures_payload = fetch_json(f"{BASE_URL}/fixtures/")
    if fixtures_payload is None:
        raise RuntimeError("Unable to load FPL fixtures data.")

    current_event = int(bootstrap.get("current_event", 1))
    upcoming_fixtures = [
        fixture for fixture in fixtures_payload
        if fixture.get("event") is not None
        and fixture.get("finished") is False
        and current_event <= int(fixture["event"]) <= current_event + next_gameweeks - 1
    ]

    rows: List[Dict[str, Any]] = []
    for fixture in upcoming_fixtures:
        event = fixture.get("event")
        team_h_id = fixture.get("team_h")
        team_a_id = fixture.get("team_a")
        opponent_h = fixture.get("team_a")
        opponent_a = fixture.get("team_h")

        rows.append({
            "event": event,
            "team_id": team_h_id,
            "team_name": team_map.get(team_h_id),
            "opponent_team_id": opponent_h,
            "opponent_team_name": team_map.get(opponent_h),
            "is_home": True,
            "fdr": fixture.get("team_h_difficulty"),
            "difficulty": fixture.get("team_h_difficulty"),
        })
        rows.append({
            "event": event,
            "team_id": team_a_id,
            "team_name": team_map.get(team_a_id),
            "opponent_team_id": opponent_a,
            "opponent_team_name": team_map.get(opponent_a),
            "is_home": False,
            "fdr": fixture.get("team_a_difficulty"),
            "difficulty": fixture.get("team_a_difficulty"),
        })

    fixtures_df = pd.DataFrame(rows)
    if not fixtures_df.empty:
        fixtures_df = fixtures_df[[
            "event",
            "team_id",
            "team_name",
            "opponent_team_id",
            "opponent_team_name",
            "is_home",
            "fdr",
            "difficulty",
        ]]

    return fixtures_df


def get_player_history(player_id: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Fetch a player's current-season and historical-season summary data."""
    api_url = f"{BASE_URL}/element-summary/{player_id}/"
    payload = fetch_json(api_url)
    if payload is None:
        return pd.DataFrame(), pd.DataFrame()

    history_df = pd.DataFrame(payload.get("history", []))
    history_past_df = pd.DataFrame(payload.get("history_past", []))

    if not history_df.empty:
        history_df["player_id"] = int(player_id)
    if not history_past_df.empty:
        history_past_df["player_id"] = int(player_id)

    return history_df, history_past_df


def get_multiple_players_history(player_ids: Iterable[int]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Fetch and concatenate history for several players with a short pause between requests."""
    all_history: List[pd.DataFrame] = []
    all_history_past: List[pd.DataFrame] = []

    for idx, player_id in enumerate(player_ids):
        if idx > 0:
            time.sleep(0.35)

        history_df, history_past_df = get_player_history(int(player_id))

        if not history_df.empty:
            history_df = history_df.copy()
            history_df["player_id"] = int(player_id)
            all_history.append(history_df)

        if not history_past_df.empty:
            history_past_df = history_past_df.copy()
            history_past_df["player_id"] = int(player_id)
            all_history_past.append(history_past_df)

    combined_history = pd.concat(all_history, ignore_index=True) if all_history else pd.DataFrame()
    combined_history_past = pd.concat(all_history_past, ignore_index=True) if all_history_past else pd.DataFrame()
    return combined_history, combined_history_past


# Load the core data and preview it
players_df, teams_df, positions_df = build_core_dataframes()
fixtures_df = load_fixtures(next_gameweeks=5)

sample_player_id = int(players_df["player_id"].iloc[0]) if not players_df.empty else 1
history_df, history_past_df = get_player_history(sample_player_id)

print("players_df.head()")
print(players_df.head())
print("\n")

print("teams_df.head()")
print(teams_df.head())
print("\n")

print("positions_df.head()")
print(positions_df.head())
print("\n")

print("fixtures_df.head()")
print(fixtures_df.head())
print("\n")

print("history_df.head()")
print(history_df.head())
print("\n")

# Example of using the combined-history helper on a shortlist of players
# This is intentionally throttled to reduce API load.
# example_player_ids = players_df["player_id"].head(5).tolist()
# combined_history, combined_history_past = get_multiple_players_history(example_player_ids)
# print(combined_history.head())

players_df.head()
   player_id          name  team_id     team  position_id position  price  \
0          1          Raya        1  Arsenal            1      GKP    6.0   
1          2  Arrizabalaga        1  Arsenal            1      GKP    5.0   
2          3       Meslier        1  Arsenal            1      GKP    5.0   
3          4       Gabriel        1  Arsenal            2      DEF    8.0   
4          5      J.Timber        1  Arsenal            2      DEF    6.5   

   total_points form    xg    xa  ict_index  minutes  ownership_pct status  \
0            12  6.0  0.00  0.00        1.8      180           37.9      a   
1             0  0.0  0.00  0.00        0.0        0            0.1      a   
2             0  0.0  0.00  0.00        0.0        0            0.0      a   
3            13  6.5  0.18  0.03        3.6      180           26.7      a   
4             0  0.0  0.00  0.00        0.0        0            0.1      i   

  availability  
0            a  
1            a  

In [ ]:
# ### 2. Initial Data Checks
#
# Before moving into analysis, we need to confirm the data loaded correctly
# and identify missing values, duplicates, or mapping issues.


SyntaxError: invalid syntax (1245419575.py, line 3)

In [ ]:

### **2.1 Quick shape and structure checks**

import pandas as pd

frames = {
    "players": players_df,
    "teams": teams_df,
    "positions": positions_df,
    "fixtures": fixtures_df,
    "history": history_df,
}

print("DataFrame shapes:")
for name, df in frames.items():
    print(f"{name}: {df.shape}")
print("\n")

print("Column dtypes:")
for name, df in frames.items():
    print(f"\n{name}:")
    print(df.dtypes)
print("\n")


In [ ]:
### **2.2 Missing values check**

print("Missing values by dataframe:\n")
for name, df in frames.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f"{name}: no missing values")
    else:
        print(f"{name}:\n{missing}\n")


In [ ]:
### **2.3 Duplicate ID checks**

print("Duplicate ID checks:\n")
for name, key in {
    "players": "player_id",
    "teams": "team_id",
    "positions": "position_id",
    "fixtures": "team_id",
    "history": "player_id",
}.items():
    df = frames.get(name)
    if df is not None and key in df.columns:
        dup_count = int(df[key].duplicated().sum())
        print(f"{name}[{key}] duplicate rows: {dup_count}")


In [ ]:
### **2.4 Integrity checks for key mappings**

print("\nIntegrity checks:\n")

players_missing_team = players_df[players_df["team"].isna()]
players_missing_position = players_df[players_df["position"].isna()]
if players_missing_team.empty:
    print("players_df team mapping: OK")
else:
    print(f"players_df with missing team mapping: {players_missing_team.shape[0]} rows")

if players_missing_position.empty:
    print("players_df position mapping: OK")
else:
    print(f"players_df with missing position mapping: {players_missing_position.shape[0]} rows")

fixtures_missing_team = fixtures_df[fixtures_df["team_name"].isna() | fixtures_df["opponent_team_name"].isna()]
if fixtures_missing_team.empty:
    print("fixtures_df team mapping: OK")
else:
    print(f"fixtures_df with missing team names: {fixtures_missing_team.shape[0]} rows")


In [ ]:
### **2.5 Numeric validation for key analytical columns**

numeric_checks = {
    "players": ["price", "total_points", "form", "xg", "xa", "ict_index", "minutes", "ownership_pct"],
    "fixtures": ["event", "fdr", "difficulty"],
    "history": ["total_points", "minutes", "goals_scored", "assists", "clean_sheets", "goals_conceded"],
}

print("\nNumeric sanity checks:\n")
for name, cols in numeric_checks.items():
    df = frames.get(name)
    if df is None:
        continue
    available = [c for c in cols if c in df.columns]
    if not available:
        continue
    print(f"{name} checked fields: {available}")
    for col in available:
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f"  {col}: min={df[col].min()}, max={df[col].max()}, mean={df[col].mean():.2f}")
        else:
            print(f"  {col}: non-numeric dtype ({df[col].dtype})")

print("\nInitial data checks complete.")
print("Review any missing values or broken mappings before moving deeper into analysis.")
